# Growth baseline · 24 июля 2026

Воспроизводимый агрегированный срез production-воронки Академического Салона. В наборе нет IP, контактов, текстов работ, query string, идентификаторов заказов или браузеров. Источник — только групповые счётчики из `visits`, `orders` и `funnel_events`.

Важно: до релиза 24 июля `visits.order_id` фактически не заполнялся, поэтому историческую конверсию session → order считать нельзя. Новая append-only воронка считается базовой только после выкладки фронтенда.

In [ ]:
import pandas as pd

daily = pd.DataFrame([
    {"day": "2026-07-15", "sessions": 14, "human_sessions": 14, "human_pageviews": 45},
    {"day": "2026-07-16", "sessions": 53, "human_sessions": 48, "human_pageviews": 228},
    {"day": "2026-07-17", "sessions": 77, "human_sessions": 61, "human_pageviews": 262},
    {"day": "2026-07-18", "sessions": 25, "human_sessions": 23, "human_pageviews": 128},
    {"day": "2026-07-19", "sessions": 28, "human_sessions": 20, "human_pageviews": 96},
    {"day": "2026-07-20", "sessions": 22, "human_sessions": 14, "human_pageviews": 37},
    {"day": "2026-07-21", "sessions": 48, "human_sessions": 41, "human_pageviews": 153},
    {"day": "2026-07-22", "sessions": 44, "human_sessions": 37, "human_pageviews": 176},
    {"day": "2026-07-23", "sessions": 39, "human_sessions": 37, "human_pageviews": 256},
    {"day": "2026-07-24", "sessions": 3, "human_sessions": 3, "human_pageviews": 24},
])
daily["day"] = pd.to_datetime(daily["day"])
daily["pages_per_human_session"] = daily["human_pageviews"] / daily["human_sessions"]
daily

In [ ]:
sources = pd.DataFrame([
    {"source": "direct_or_unknown", "human_sessions": 241, "pageviews": 1144},
    {"source": "other_referral", "human_sessions": 40, "pageviews": 167},
    {"source": "yandex", "human_sessions": 14, "pageviews": 90},
    {"source": "google", "human_sessions": 3, "pageviews": 4},
])
sources["session_share"] = sources["human_sessions"] / sources["human_sessions"].sum()
sources

In [ ]:
summary = pd.Series({
    "human_sessions": int(daily.human_sessions.sum()),
    "human_pageviews": int(daily.human_pageviews.sum()),
    "pages_per_session": round(daily.human_pageviews.sum() / daily.human_sessions.sum(), 2),
    "known_search_sessions": int(sources.loc[sources.source.isin(["yandex", "google"]), "human_sessions"].sum()),
    "known_search_share_pct": round(100 * sources.loc[sources.source.isin(["yandex", "google"]), "human_sessions"].sum() / sources.human_sessions.sum(), 1),
})
summary

## Интерпретация и семидневные KPI

- 298 человеческих сессий и 1 405 просмотров за неполные 10 дней: охват уже есть, но известный поисковый трафик составляет только 17 сессий (5,7%).
- 80,9% трафика попадает в `direct_or_unknown`; это одновременно реальный direct и потерянная атрибуция. Все новые owned-ссылки поэтому получают UTM.
- Исторические заказы нельзя честно делить на эти сессии: прежняя связь событий с заказом была сломана, а часть записей удалена как тестовая/невалидная.
- Операционный таргет на 7 дней после релиза: ≥300 human sessions, ≥25 известных search sessions, ≥3 валидных `submit_ok`; CTA → first_input ≥25%, first_input → submit_ok ≥15%.
- Guardrails: рост `validation_error / submit_attempt`, `submit_fail / submit_attempt` или доли ботов не считать успехом; тексты и контакты в аналитику не добавлять.